# Assignment 6: Transfer Learning with VGG16 (Flowers)
Change the folder path to `flower_photos` on the pendrive (one sub-folder per flower).

In [1]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    'flower_photos', validation_split=0.2, subset='training', seed=42, image_size=(150, 150))
test_ds = tf.keras.utils.image_dataset_from_directory(
    'flower_photos', validation_split=0.2, subset='validation', seed=42, image_size=(150, 150))

class_names = train_ds.class_names
print(class_names)

train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y))
test_ds = test_ds.map(lambda x, y: (preprocess_input(x), y))

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
base_model.trainable = False

model = Sequential()

model.add(base_model)
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(50, activation='relu'))
model.add(tf.keras.layers.Dense(20, activation='relu'))
model.add(tf.keras.layers.Dense(5, activation='softmax'))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()

In [ ]:
H = model.fit(train_ds, validation_data=test_ds, epochs=5)

In [ ]:
base_model.trainable = True
for layer in base_model.layers:
    layer.trainable = layer.name.startswith('block5')     # unfreeze only the last block

model.compile(optimizer=tf.keras.optimizers.Adam(0.00001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

H2 = model.fit(train_ds, validation_data=test_ds, epochs=2)

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)

print('loss = %.3f' % test_loss)
print('accuracy = %.3f' % test_accuracy)

In [ ]:
images, labels = next(iter(test_ds))
prediction = model.predict(images)

for i in range(5):
    print('actual:', class_names[labels[i]], '| predicted:', class_names[np.argmax(prediction[i])])

In [ ]:
plt.plot(H.history['loss'], label='train_loss')
plt.plot(H.history['val_loss'], label='val_loss')
plt.plot(H.history['accuracy'], label='train_accuracy')
plt.plot(H.history['val_accuracy'], label='val_accuracy')
plt.title('Training Loss and Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Loss/Accuracy')
plt.legend()
plt.show()